# CS 131 Project Assignment 3 - PySpark, Parquet, and DuckDB (Team 9)

Dataset: UCI Online Retail II (both workbook sheets combined into one TSV)

Team roles this sprint:
- Kanishka Yadav (Data Engineer)
- Sanjini Shukla (Data Engineer)
- Lipi Himanshukumar Makwana (Product Manager)
- Kushan Mehta (Data Storyteller)

Setup, requirements (`pa3/requirements.txt`), and run instructions are in the repository
[README](../../README.md#project-assignment-3-pyspark-parquet-duckdb). Run from `pa3/notebooks/` with the
**PA3 (PySpark)** kernel and Java 17 or 21. Set `PA3_DATA_PATH` to the combined TSV if it is not in one of the
default locations checked in Section 0a.


## 0. Environment setup and versions

Creates a local Spark session (`local[*]`, one machine) and prints the Spark, DuckDB, and Python versions.

- The session time zone is UTC, so timestamps are parsed the same way on every machine.
- ANSI mode is off, so an invalid value becomes `null` when cast and is counted in Section 3a instead of
  stopping the run. (Spark 4.x turns ANSI mode on by default.)
- Spark's Java process writes its startup messages and logs to `pa3/logs/spark_jvm.log` (gitignored) instead of
  the notebook. These are harmless warnings from Java 21 and Spark's default logging. If Spark fails to start,
  the end of that log is printed so the real error is still visible.
- The console progress bars are turned off, so cell outputs show only results.


In [1]:
# Import the Spark and Python libraries used in this notebook.
from pyspark.sql import SparkSession
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql import types as T
from pathlib import Path
import os
import sys
import platform
import contextlib
import pandas as pd
import duckdb

# Use the same Python executable for Spark workers and the notebook kernel.
os.environ.setdefault("PYSPARK_PYTHON", sys.executable)

# Send the Spark JVM's console output (Java warnings, log4j startup lines) to a log file
# instead of the notebook. The JVM keeps the file it was started with, so later JVM log
# lines also go there; Python still raises any Spark error in the cell as usual.
LOG_DIR = Path.cwd().parent / "logs"
LOG_DIR.mkdir(parents=True, exist_ok=True)
SPARK_LOG = LOG_DIR / "spark_jvm.log"


@contextlib.contextmanager
def jvm_output_to(log_path):
    sys.stderr.flush()
    saved_stderr = os.dup(2)
    with open(log_path, "w") as log_file:
        os.dup2(log_file.fileno(), 2)
    try:
        yield
    finally:
        os.dup2(saved_stderr, 2)
        os.close(saved_stderr)


# Create a local Spark session for this assignment.
try:
    with jvm_output_to(SPARK_LOG):
        spark = (
            SparkSession.builder
            .appName("CS131-PA3-Online-Retail")
            .master("local[*]")
            .config("spark.sql.shuffle.partitions", "8")
            .config("spark.sql.session.timeZone", "UTC")
            .config("spark.sql.ansi.enabled", "false")
            .config("spark.ui.showConsoleProgress", "false")
            .getOrCreate()
        )
except Exception:
    print(f"Spark failed to start. Last lines of {SPARK_LOG}:")
    print("".join(SPARK_LOG.read_text().splitlines(keepends=True)[-30:]))
    raise

spark.sparkContext.setLogLevel("ERROR")

print("Spark version:", spark.version)
print("DuckDB version:", duckdb.__version__)
print("Python version:", platform.python_version())
print("Spark master:", spark.sparkContext.master, f"({spark.sparkContext.defaultParallelism} cores)")
print("Spark ran locally:", spark.sparkContext.master.startswith("local"))
print("Spark JVM log:", SPARK_LOG)


Spark version: 4.3.0
DuckDB version: 2.0.1
Python version: 3.12.14
Spark master: local[*] (10 cores)
Spark ran locally: True
Spark JVM log: /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/logs/spark_jvm.log


### 0a. Paths

Run from `pa3/notebooks/`. The input is the first one found:

1. `PA3_DATA_PATH`, if set (use this for the full TSV in a custom location).
2. The full combined TSV in a default location outside the repo.
3. The committed 1,000-row sample, `pa3/data/sample/online_retail_sample.tsv`, for development or for running the
   notebook without the full dataset.

A sample run writes to `pa3/out/sample_run/` and `pa3/data/parquet/sample_run/` (both gitignored), so it never
overwrites the full-data outputs in `pa3/out/`.

On IBM, if the TSV sits next to the repository folder (as in PA2), no setting is needed: default 2 finds it.
Otherwise, set an **absolute** path before starting Jupyter (a relative path is resolved from `pa3/notebooks/`):

```bash
export PA3_DATA_PATH="$HOME/online_retail_II_combined.tsv"
```

In [2]:
# Define paths relative to the pa3 directory.
PA3_DIR = Path.cwd().parent
REPO_DIR = PA3_DIR.parent

SAMPLE_PATH = PA3_DIR / "data" / "sample" / "online_retail_sample.tsv"
FULL_ROWS = 1_067_371

# Use PA3_DATA_PATH when the full dataset is stored in a custom location.
DATA_PATH = os.environ.get("PA3_DATA_PATH")

if DATA_PATH:
    DATA_PATH = Path(DATA_PATH)
else:
    candidates = [
        REPO_DIR.parent / "online_retail_II_combined.tsv",
        REPO_DIR.parent / "data" / "online_retail_II_combined.tsv",
        SAMPLE_PATH,
    ]
    DATA_PATH = next((p for p in candidates if p.exists()), None)

if DATA_PATH is not None and not DATA_PATH.exists():
    raise FileNotFoundError(f"PA3_DATA_PATH is set to {DATA_PATH}, but that file does not exist.")

if DATA_PATH is None:
    raise FileNotFoundError(
        "Set PA3_DATA_PATH to the combined Online Retail II TSV file, "
        f"or run from pa3/notebooks/ so the committed sample ({SAMPLE_PATH}) is found."
    )

USING_SAMPLE = DATA_PATH.resolve() == SAMPLE_PATH.resolve()

OUT_DIR = PA3_DIR / "out"
PARQUET_DIR = PA3_DIR / "data" / "parquet" / "online_retail_prepared"
DEV_SAMPLE_DIR = PA3_DIR / "data" / "dev_sample_20pct"

# Keep sample-run files separate from the full-data deliverables.
if USING_SAMPLE:
    OUT_DIR = PA3_DIR / "out" / "sample_run"
    PARQUET_DIR = PA3_DIR / "data" / "parquet" / "sample_run" / "online_retail_prepared"
    DEV_SAMPLE_DIR = PA3_DIR / "data" / "parquet" / "sample_run" / "dev_sample_20pct"

OUT_DIR.mkdir(parents=True, exist_ok=True)

print("Input:", DATA_PATH)
print("Input size (bytes):", f"{DATA_PATH.stat().st_size:,}")
print("Using committed 1,000-row sample:", USING_SAMPLE)
print("Outputs:", OUT_DIR)


Input: /Users/kanishka/CS_131/project/data/online_retail_II_combined.tsv
Input size (bytes): 93,170,518
Using committed 1,000-row sample: False
Outputs: /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/out


## 1. Project goal, questions, and data description

**Lipi:** To be completed. Include the project goal, the two questions, row meaning, key, important columns,
data types, and units. The data types must match the schema contract in Section 3b.

## 2. Read the data with PySpark

Reads the combined Online Retail II TSV (1,067,371 transaction rows plus a header) and shows its schema.

- **Explicit string schema:** all eight columns are read as strings. Section 3 casts them, so cast failures can be counted.
- **`quote=""`:** quoting is off because some descriptions contain literal `"` characters. Each line is split on
  tabs, as in PA2.
- **`mode="FAILFAST"`:** the read stops if any line does not have eight fields.
- Empty fields are read as `null`.

In [3]:
# Read the TSV with an explicit string schema.
RAW_COLUMNS = [
    "Invoice",
    "StockCode",
    "Description",
    "Quantity",
    "InvoiceDate",
    "Price",
    "Customer ID",
    "Country",
]

raw_schema = T.StructType([
    T.StructField(column, T.StringType(), True)
    for column in RAW_COLUMNS
])

raw_df = (
    spark.read
    .option("header", True)
    .option("sep", "\t")
    .option("quote", "")
    .option("mode", "FAILFAST")
    .schema(raw_schema)
    .csv(str(DATA_PATH))
)

raw_rows = raw_df.count()
print("Rows:", raw_rows)

if USING_SAMPLE:
    assert raw_rows == 1_000, f"Expected 1,000 sample rows, got {raw_rows:,}"
else:
    assert raw_rows == FULL_ROWS, f"Expected {FULL_ROWS:,} rows, got {raw_rows:,}"

raw_df.printSchema()
raw_df.show(5, truncate=False)


Rows: 1067371
root
 |-- Invoice: string (nullable = true)
 |-- StockCode: string (nullable = true)
 |-- Description: string (nullable = true)
 |-- Quantity: string (nullable = true)
 |-- InvoiceDate: string (nullable = true)
 |-- Price: string (nullable = true)
 |-- Customer ID: string (nullable = true)
 |-- Country: string (nullable = true)



+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|Invoice|StockCode|Description                        |Quantity|InvoiceDate        |Price|Customer ID|Country       |
+-------+---------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048    |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085      |United Kingdom|
|489434 |79323P   |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75 |13085      |United Kingdom|
|489434 |79323W   | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75 |13085      |United Kingdom|
|489434 |22041    |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2009-12-01 07:45:00|2.1  |13085      |United Kingdom|
|489434 |21232    |STRAWBERRY CERAMIC TRINKET BOX     |24      |2009-12-01 07:45:00|1.25 |13085      |United Kingdom|
+-------+---------+-----------------------------------+-

## 3. Select and cast columns

Renames the columns to snake_case and casts them to analysis types. Text and ID columns are not trimmed or
re-cased: `stock_code` stays case-sensitive (`85123A` and `85123a` are different codes) and untrimmed, matching
how PA2 counted stock codes.

| Column | Source | Type | Unit / meaning |
|---|---|---|---|
| `invoice` | Invoice | string | Invoice number; a leading `C` marks a cancellation |
| `stock_code` | StockCode | string (unchanged) | Product code |
| `description` | Description | string | Product name |
| `quantity` | Quantity | integer | Units of the product on this invoice line |
| `invoice_date` | InvoiceDate | timestamp (`yyyy-MM-dd HH:mm:ss`) | Invoice date and time |
| `price` | Price | double | Price per unit, pounds sterling (GBP) |
| `customer_id` | Customer ID | string | Customer identifier (an ID, not a measure) |
| `country` | Country | string (unchanged) | Customer's country |

**Row meaning:** one row is one invoice line (one product on one invoice).

In [4]:
# Select the needed columns and cast them to analysis types.
typed_df = raw_df.select(
    F.col("Invoice").alias("invoice"),
    F.col("StockCode").alias("stock_code"),
    F.col("Description").alias("description"),
    F.col("Quantity").cast("int").alias("quantity"),
    F.to_timestamp(
        F.col("InvoiceDate"),
        "yyyy-MM-dd HH:mm:ss"
    ).alias("invoice_date"),
    F.col("Price").cast("double").alias("price"),
    F.col("Customer ID").alias("customer_id"),
    F.col("Country").alias("country"),
)

typed_df.printSchema()
typed_df.show(5, truncate=False)


root
 |-- invoice: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)



+-------+----------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|invoice|stock_code|description                        |quantity|invoice_date       |price|customer_id|country       |
+-------+----------+-----------------------------------+--------+-------------------+-----+-----------+--------------+
|489434 |85048     |15CM CHRISTMAS GLASS BALL 20 LIGHTS|12      |2009-12-01 07:45:00|6.95 |13085      |United Kingdom|
|489434 |79323P    |PINK CHERRY LIGHTS                 |12      |2009-12-01 07:45:00|6.75 |13085      |United Kingdom|
|489434 |79323W    | WHITE CHERRY LIGHTS               |12      |2009-12-01 07:45:00|6.75 |13085      |United Kingdom|
|489434 |22041     |"RECORD FRAME 7"" SINGLE SIZE "    |48      |2009-12-01 07:45:00|2.1  |13085      |United Kingdom|
|489434 |21232     |STRAWBERRY CERAMIC TRINKET BOX     |24      |2009-12-01 07:45:00|1.25 |13085      |United Kingdom|
+-------+----------+----------------------------

### 3a. Cast failures and missing key values

A **cast failure** is a value that is present in the raw data but `null` after casting. A **blank** is a string
that is empty after trimming. Both are counted; no values are changed. PySpark and DuckDB apply the same rules
because both read the same Parquet.

| Case | Rule |
|---|---|
| Null or blank `stock_code` or `country` | Expected 0 (asserted). If any appeared, Analysis 1 would keep them as their own group, as PA2's `uniq -c` did. `Unspecified` is a real country value. |
| Invalid `quantity`, `price`, or `invoice_date` | Expected 0 cast failures (asserted). |
| Missing `customer_id` | Kept and flagged in Section 5. |

In [5]:
# Count values that fail during casting or are missing from the raw input.
checks = raw_df.agg(
    F.sum(F.when(F.col("Quantity").isNotNull() & F.col("Quantity").cast("int").isNull(), 1).otherwise(0)).alias("quantity_cast_failures"),
    F.sum(F.when(F.col("Price").isNotNull() & F.col("Price").cast("double").isNull(), 1).otherwise(0)).alias("price_cast_failures"),
    F.sum(F.when(F.col("InvoiceDate").isNotNull() & F.to_timestamp(F.col("InvoiceDate"), "yyyy-MM-dd HH:mm:ss").isNull(), 1).otherwise(0)).alias("invoice_date_cast_failures"),
    F.sum(F.when(F.col("Quantity").isNull(), 1).otherwise(0)).alias("null_quantity"),
    F.sum(F.when(F.col("Price").isNull(), 1).otherwise(0)).alias("null_price"),
    F.sum(F.when(F.col("InvoiceDate").isNull(), 1).otherwise(0)).alias("null_invoice_date"),
    F.sum(F.when(F.col("StockCode").isNull(), 1).otherwise(0)).alias("null_stock_code"),
    F.sum(F.when(F.trim(F.col("StockCode")) == "", 1).otherwise(0)).alias("blank_stock_code"),
    F.sum(F.when(F.col("Country").isNull(), 1).otherwise(0)).alias("null_country"),
    F.sum(F.when(F.trim(F.col("Country")) == "", 1).otherwise(0)).alias("blank_country"),
    F.sum(F.when(F.col("Customer ID").isNull() | (F.trim(F.col("Customer ID")) == ""), 1).otherwise(0)).alias("missing_customer_id"),
    F.sum(F.when(F.col("StockCode") != F.trim(F.col("StockCode")), 1).otherwise(0)).alias("stock_code_with_edge_whitespace"),
).first().asDict()

for name, value in checks.items():
    print(f"{name:34s} {value:>10,}")

must_be_zero = [
    "quantity_cast_failures", "price_cast_failures", "invoice_date_cast_failures",
    "null_quantity", "null_price", "null_invoice_date",
    "null_stock_code", "blank_stock_code", "null_country", "blank_country",
]
nonzero = {name: checks[name] for name in must_be_zero if checks[name] != 0}
assert not nonzero, f"Unexpected missing or invalid values: {nonzero}"
print("\nAll cast-failure and missing-key checks are zero.")


quantity_cast_failures                      0
price_cast_failures                         0
invoice_date_cast_failures                  0
null_quantity                               0
null_price                                  0
null_invoice_date                           0
null_stock_code                             0
blank_stock_code                            0
null_country                                0
blank_country                               0
missing_customer_id                   243,007
stock_code_with_edge_whitespace             1

All cast-failure and missing-key checks are zero.


### 3b. Schema contract for downstream sections

Sections 4–8 add columns to this schema. **Sections 9–12 must use `prepared`, the DataFrame reloaded from Parquet
in Section 7**, not `typed_df`.

| Column | Type | Added in | Notes |
|---|---|---|---|
| `invoice` | string | 3 | Unchanged |
| `stock_code` | string | 3 | **Unchanged — do not trim or change case** |
| `description` | string | 3 | Literal text (quoting was off, so any `"` characters are kept) |
| `quantity` | int | 3 | Units |
| `invoice_date` | timestamp | 3 | Session time zone is UTC |
| `price` | double | 3 | GBP per unit |
| `customer_id` | string | 3 | `null` when missing |
| `country` | string | 3 | Unchanged |
| `revenue` | double | 4 | `quantity * price`, GBP; `null` if either input is `null`; negative for returns |
| `is_cancellation` | boolean | 4 | `invoice` starts with `C` |
| `flag_missing_customer` | boolean | 5 | `customer_id` is null or blank |
| `flag_bad_qty_price` | boolean | 5 | `quantity` or `price` is `null` or ≤ 0 |

Parquet location (generated locally, gitignored): `pa3/data/parquet/online_retail_prepared/`.

## 4. Derived columns

Adds two columns used by the analyses. Nothing is rounded here; rounding happens only in exported results.

| Column | Type | Rule |
|---|---|---|
| `revenue` | double | `quantity * price` in GBP; `null` if either input is `null`; negative for returns |
| `is_cancellation` | boolean | `invoice` starts with `C` |

In [6]:
# Add the derived revenue and cancellation columns.
derived_df = (
    typed_df
    .withColumn("revenue", F.col("quantity") * F.col("price"))
    .withColumn("is_cancellation", F.col("invoice").startswith("C"))
)

derived_df.select("invoice", "stock_code", "quantity", "price", "revenue", "is_cancellation") \
    .orderBy(F.col("is_cancellation").desc(), "invoice") \
    .show(5, truncate=False)

# Check the derived revenue value on a few rows.
for row in derived_df.select("quantity", "price", "revenue").limit(5).collect():
    assert abs(row["quantity"] * row["price"] - row["revenue"]) < 1e-9
print("revenue spot check passed")

+-------+----------+--------+-----+-------------------+---------------+
|invoice|stock_code|quantity|price|revenue            |is_cancellation|
+-------+----------+--------+-----+-------------------+---------------+
|C489449|22087     |-12     |2.95 |-35.400000000000006|true           |
|C489449|21871     |-12     |1.25 |-15.0              |true           |
|C489449|85206A    |-6      |1.65 |-9.899999999999999 |true           |
|C489449|21895     |-4      |4.25 |-17.0              |true           |
|C489449|21896     |-6      |2.1  |-12.600000000000001|true           |
+-------+----------+--------+-----+-------------------+---------------+
only showing top 5 rows


revenue spot check passed


## 5. Data-quality checks

Adds two quality flags and counts the affected rows. No rows are removed.

**Key:** the data has no natural key. There is no line-item ID, and `(invoice, stock_code)` repeats on some
invoices (counted below).

| Check | Rule | Decision |
|---|---|---|
| A. Missing customer | `customer_id` is null or blank | Kept and flagged (`flag_missing_customer`). These are real sales without a customer account; customer-level work must filter them out. |
| B. Bad quantity or price | `quantity` or `price` is null or ≤ 0 | Kept and flagged (`flag_bad_qty_price`). Mostly cancellations and returns. PA2 Part 1's stock-code counts included them, so Analysis 1 keeps them. Use `NOT flag_bad_qty_price` for completed purchases only: this is PA2 Part 2's filter, which removed the same 25,700 rows. |

**Overlap:** rows with at least one problem = A + B − (A and B), not A + B. Both the overlap and the union are reported.

**Exact duplicates are kept.** Without a line-item ID, identical rows may be the same product scanned twice on
one invoice. They are counted as duplicate groups (PA2's 32,907 figure) and as extra copies.

In [7]:
# Add quality flags without removing rows.
flagged_df = (
    derived_df
    .withColumn(
        "flag_missing_customer",
        F.col("customer_id").isNull()
        | (F.trim(F.col("customer_id")) == ""),
    )
    .withColumn(
        "flag_bad_qty_price",
        F.col("quantity").isNull() | F.col("price").isNull()
        | (F.col("quantity") <= 0) | (F.col("price") <= 0),
    )
)

total_rows = flagged_df.count()

flag_counts = flagged_df.agg(
    F.sum(F.col("flag_missing_customer").cast("int")).alias("a"),
    F.sum(F.col("flag_bad_qty_price").cast("int")).alias("b"),
    F.sum((F.col("flag_missing_customer") & F.col("flag_bad_qty_price")).cast("int")).alias("a_and_b"),
    F.sum((F.col("flag_missing_customer") | F.col("flag_bad_qty_price")).cast("int")).alias("a_or_b"),
    F.sum(F.col("is_cancellation").cast("int")).alias("cancel"),
    F.sum((F.col("is_cancellation") & F.col("flag_bad_qty_price")).cast("int")).alias("cancel_and_b"),
).first()

# Subtract the overlap once when calculating the union.
assert flag_counts["a_or_b"] == flag_counts["a"] + flag_counts["b"] - flag_counts["a_and_b"]

# Count repeated complete rows without deleting them.
dup_groups = (
    typed_df.groupBy(typed_df.columns).count()
    .filter(F.col("count") > 1)
    .agg(F.count("*").alias("groups"), F.sum(F.col("count") - 1).alias("extra_copies"))
    .first()
)
dup_group_count = dup_groups["groups"] or 0
dup_extra_copies = dup_groups["extra_copies"] or 0
assert total_rows - typed_df.dropDuplicates().count() == dup_extra_copies

# Check whether invoice and stock_code form a unique key.
pair_rows = (
    typed_df.groupBy("invoice", "stock_code").count()
    .filter(F.col("count") > 1)
    .agg(F.sum("count").alias("rows"))
    .first()["rows"] or 0
)

for name, value in [
    ("total rows", total_rows),
    ("A. missing customer_id", flag_counts["a"]),
    ("B. quantity/price null or <= 0", flag_counts["b"]),
    ("A and B (overlap)", flag_counts["a_and_b"]),
    ("A or B (union)", flag_counts["a_or_b"]),
    ("cancellations (invoice starts with C)", flag_counts["cancel"]),
    ("cancellations that also fail B", flag_counts["cancel_and_b"]),
    ("exact duplicate groups", dup_group_count),
    ("exact duplicate extra copies", dup_extra_copies),
    ("rows sharing (invoice, stock_code)", pair_rows),
]:
    print(f"{name:40s} {value:>10,}")

total rows                                1,067,371
A. missing customer_id                      243,007
B. quantity/price null or <= 0               25,700
A and B (overlap)                             6,885
A or B (union)                              261,822
cancellations (invoice starts with C)        19,494
cancellations that also fail B               19,493
exact duplicate groups                       32,907
exact duplicate extra copies                 34,335
rows sharing (invoice, stock_code)           88,585


### 5a. Save `out/quality_checks.csv`

One row per check, with its share of all rows. `overlap_note` shows where checks overlap, so the counts are not
simply added.

In [8]:
def pct(n):
    return round(100.0 * n / total_rows, 2) if total_rows else 0.0

quality_rows = [
    ("Q0", "total_rows", "All rows read from the input", total_rows,
     "Population for every percentage below", ""),
    ("Q1", "cast_failures", "Raw quantity, price, or invoice_date present but null after casting",
     checks["quantity_cast_failures"] + checks["price_cast_failures"] + checks["invoice_date_cast_failures"],
     "Asserted to be 0 (Section 3a)", ""),
    ("Q2", "missing_customer_id", "customer_id is null or blank", flag_counts["a"],
     "Kept; flag_missing_customer = true", f"{flag_counts['a_and_b']:,} of these rows also fail Q3"),
    ("Q3", "nonpositive_or_missing_qty_price", "quantity or price is null or <= 0", flag_counts["b"],
     "Kept; flag_bad_qty_price = true", f"{flag_counts['a_and_b']:,} of these rows also fail Q2"),
    ("Q4", "overlap_q2_and_q3", "Rows failing both Q2 and Q3", flag_counts["a_and_b"],
     "Informational", "Counted in both Q2 and Q3"),
    ("Q5", "union_q2_or_q3", "Rows failing Q2 or Q3 (at least one)", flag_counts["a_or_b"],
     "Informational", "Q2 + Q3 - Q4; not Q2 + Q3"),
    ("Q6", "cancellation_rows", "invoice starts with 'C'", flag_counts["cancel"],
     "Kept; is_cancellation = true", f"{flag_counts['cancel_and_b']:,} of these rows also fail Q3"),
    ("Q7", "exact_duplicate_extra_copies", "Rows identical in all 8 columns to an earlier row", dup_extra_copies,
     "Kept; cannot prove true duplicates without a line-item ID",
     f"Spread over {dup_group_count:,} duplicate groups (PA2 reported the group count)"),
    ("Q8", "rows_sharing_invoice_stock_code", "(invoice, stock_code) appears on more than one row", pair_rows,
     "Informational; shows (invoice, stock_code) is not a key", "Includes all Q7 rows"),
    ("Q9", "stock_code_edge_whitespace", "stock_code has leading or trailing whitespace",
     checks["stock_code_with_edge_whitespace"], "Kept unchanged to match PA2 grouping", ""),
]

quality_pdf = pd.DataFrame(
    quality_rows,
    columns=["check_id", "check_name", "rule", "rows_affected", "action", "overlap_note"],
)
quality_pdf.insert(4, "pct_of_rows", quality_pdf["rows_affected"].map(pct))

quality_pdf.to_csv(OUT_DIR / "quality_checks.csv", index=False)
print("Wrote", OUT_DIR / "quality_checks.csv")
quality_pdf

Wrote /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/out/quality_checks.csv


,check_id,check_name,rule,rows_affected,pct_of_rows,action,overlap_note
0,Q0,total_rows,All rows read from the input,1067371,100.00,Population for every percentage below,
1,Q1,cast_failures,"Raw quantity, price, or invoice_date present b...",0,0.00,Asserted to be 0 (Section 3a),
2,Q2,missing_customer_id,customer_id is null or blank,243007,22.77,Kept; flag_missing_customer = true,"6,885 of these rows also fail Q3"
3,Q3,nonpositive_or_missing_qty_price,quantity or price is null or <= 0,25700,2.41,Kept; flag_bad_qty_price = true,"6,885 of these rows also fail Q2"
4,Q4,overlap_q2_and_q3,Rows failing both Q2 and Q3,6885,0.65,Informational,Counted in both Q2 and Q3
5,Q5,union_q2_or_q3,Rows failing Q2 or Q3 (at least one),261822,24.53,Informational,Q2 + Q3 - Q4; not Q2 + Q3
6,Q6,cancellation_rows,invoice starts with 'C',19494,1.83,Kept; is_cancellation = true,"19,493 of these rows also fail Q3"
7,Q7,exact_duplicate_extra_copies,Rows identical in all 8 columns to an earlier row,34335,3.22,Kept; cannot prove true duplicates without a l...,"Spread over 32,907 duplicate groups (PA2 repor..."
8,Q8,rows_sharing_invoice_stock_code,"(invoice, stock_code) appears on more than one...",88585,8.30,"Informational; shows (invoice, stock_code) is ...",Includes all Q7 rows
9,Q9,stock_code_edge_whitespace,stock_code has leading or trailing whitespace,1,0.00,Kept unchanged to match PA2 grouping,


## 6. Write Parquet

Writes the prepared data (8 cast columns, 2 derived columns, 2 flags) to Parquet, replacing any earlier run. The
Parquet folder is gitignored.

In [9]:
# Select the prepared columns and write them as Parquet.
prepared_columns = [
    "invoice", "stock_code", "description", "quantity", "invoice_date", "price",
    "customer_id", "country", "revenue", "is_cancellation",
    "flag_missing_customer", "flag_bad_qty_price",
]
prepared_to_write = flagged_df.select(*prepared_columns)

prepared_to_write.write.mode("overwrite").parquet(str(PARQUET_DIR))

parquet_bytes = sum(p.stat().st_size for p in PARQUET_DIR.rglob("*.parquet"))
print("Wrote", PARQUET_DIR)
print("Parquet size (bytes):", f"{parquet_bytes:,}")

Wrote /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/data/parquet/online_retail_prepared
Parquet size (bytes): 9,092,629


## 7. Read Parquet back and verify

Reads the Parquet back as `prepared` and checks that the row count, column order, and data types are unchanged.
Sections 8–12 use `prepared`.

In [10]:
# Read the prepared Parquet data back into a new DataFrame.
prepared = spark.read.parquet(str(PARQUET_DIR))

prepared_rows = prepared.count()
expected_types = [(f.name, f.dataType.simpleString()) for f in prepared_to_write.schema.fields]
reloaded_types = [(f.name, f.dataType.simpleString()) for f in prepared.schema.fields]

print("Rows before write:", f"{total_rows:,}")
print("Rows after reload:", f"{prepared_rows:,}")
prepared.printSchema()

assert prepared_rows == total_rows, "Row count changed after the Parquet round trip"
assert reloaded_types == expected_types, f"Schema changed:\n{expected_types}\n{reloaded_types}"
assert prepared.filter(F.col("flag_missing_customer")).count() == flag_counts["a"]
print("Parquet round trip verified: same row count, column order, and data types.")

Rows before write: 1,067,371
Rows after reload: 1,067,371
root
 |-- invoice: string (nullable = true)
 |-- stock_code: string (nullable = true)
 |-- description: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- invoice_date: timestamp (nullable = true)
 |-- price: double (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- country: string (nullable = true)
 |-- revenue: double (nullable = true)
 |-- is_cancellation: boolean (nullable = true)
 |-- flag_missing_customer: boolean (nullable = true)
 |-- flag_bad_qty_price: boolean (nullable = true)



Parquet round trip verified: same row count, column order, and data types.


## 8. 20% development sample and sample comparison

Creates the development sample from `prepared` with the class method:

```python
sample = prepared.sample(withReplacement=False, fraction=0.20, seed=131)
```

- The sample has about 20% of the rows, not exactly 20%. Spark samples each partition separately, so the same
  input and seed give the same sample only with the same partitioning (number of cores and Spark version). For
  example, a 2-core Colab run selects 213,434 rows and misses Saudi Arabia (10 rows), while the 10-core run whose
  outputs are saved here selects 213,507 rows and includes every country.
- It is for development only. Analyses 1–2 and the benchmark use the full `prepared` data.
- The comparison covers `country` (every country, so missing groups are listed), `is_cancellation`, and
  `flag_missing_customer`. `pct_point_diff` is sample % minus population %.

In [11]:
# Create the development sample using the class sampling pattern.
SAMPLE_FRACTION = 0.20
SAMPLE_SEED = 131

sample = prepared.sample(withReplacement=False, fraction=SAMPLE_FRACTION, seed=SAMPLE_SEED)

population_rows = prepared_rows
sample_rows = sample.count()
assert sample_rows > 0, "The development sample is empty"

print("Fraction:", SAMPLE_FRACTION, "| seed:", SAMPLE_SEED)
print("Population rows:", f"{population_rows:,}")
print("Sample rows:    ", f"{sample_rows:,}", f"({100 * sample_rows / population_rows:.2f}% of population)")

sample.write.mode("overwrite").parquet(str(DEV_SAMPLE_DIR))
print("Saved development sample to", DEV_SAMPLE_DIR)

Fraction: 0.2 | seed: 131
Population rows: 1,067,371
Sample rows:     213,507 (20.00% of population)


Saved development sample to /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/data/dev_sample_20pct


In [12]:
# Compare population and sample proportions for useful groups.
def compare_group(column):
    population = prepared.groupBy(column).agg(F.count("*").alias("population_rows"))
    sample_counts = sample.groupBy(column).agg(F.count("*").alias("sample_rows"))
    return (
        population.join(sample_counts, on=column, how="left")
        .fillna(0, subset=["sample_rows"])
        .select(
            F.lit(column).alias("group_column"),
            F.col(column).cast("string").alias("group_value"),
            "population_rows",
            "sample_rows",
        )
    )


comparison_pdf = pd.concat(
    [compare_group(c).toPandas() for c in ["country", "is_cancellation", "flag_missing_customer"]],
    ignore_index=True,
)
comparison_pdf["population_pct"] = (100 * comparison_pdf["population_rows"] / population_rows).round(4)
comparison_pdf["sample_pct"] = (100 * comparison_pdf["sample_rows"] / sample_rows).round(4)
comparison_pdf["pct_point_diff"] = (comparison_pdf["sample_pct"] - comparison_pdf["population_pct"]).round(4)
comparison_pdf["in_sample"] = comparison_pdf["sample_rows"] > 0
comparison_pdf.insert(0, "seed", SAMPLE_SEED)
comparison_pdf.insert(0, "fraction", SAMPLE_FRACTION)

# Deterministic order: grouping, then largest population group first, then name.
comparison_pdf = comparison_pdf.sort_values(
    ["group_column", "population_rows", "group_value"], ascending=[True, False, True]
).reset_index(drop=True)

# Check that each grouping accounts for every population and sample row.
for column, part in comparison_pdf.groupby("group_column"):
    assert part["population_rows"].sum() == population_rows, column
    assert part["sample_rows"].sum() == sample_rows, column

comparison_pdf.to_csv(OUT_DIR / "sample_comparison.csv", index=False)
print("Wrote", OUT_DIR / "sample_comparison.csv")

missing = comparison_pdf.loc[~comparison_pdf["in_sample"], ["group_column", "group_value", "population_rows"]]
if missing.empty:
    print("Every group in the population also appears in the sample.")
else:
    print("Groups present in the population but missing from the sample:")
    print(missing.to_string(index=False))

print("Largest absolute difference (percentage points):", comparison_pdf["pct_point_diff"].abs().max())
comparison_pdf[comparison_pdf["group_column"] != "country"]

Wrote /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/out/sample_comparison.csv
Every group in the population also appears in the sample.
Largest absolute difference (percentage points): 0.0464


,fraction,seed,group_column,group_value,population_rows,sample_rows,population_pct,sample_pct,pct_point_diff,in_sample
43,0.2,131,flag_missing_customer,false,824364,164799,77.2331,77.1867,-0.0464,True
44,0.2,131,flag_missing_customer,true,243007,48708,22.7669,22.8133,0.0464,True
45,0.2,131,is_cancellation,false,1047877,209604,98.1736,98.1720,-0.0016,True
46,0.2,131,is_cancellation,true,19494,3903,1.8264,1.8280,0.0016,True


## 9. Analysis 1 - stock-code frequency (PySpark)

Reproduces PA2's stock-code frequency table (`tail -n +2 | cut -f2 | sort | uniq -c | sort -nr`) with a grouped
aggregation in PySpark.

- Uses **all rows** of `prepared`, because PA2 counted every row (including cancellations and missing customers).
- `groupBy("stock_code")` and count the rows as `count`.
- Deterministic order: `count` descending, then `stock_code` ascending to break ties.
- On the full data the result must have 5,305 distinct stock codes, and the first 20 rows must match PA2's Top 20
  on both stock code and count. PA2's Top 20 has no ties, so the order must match too.
- Output: `analysis_1.csv` (every stock code, columns `stock_code, count`).


In [13]:
# Analysis 1: Count every stock code across all rows.
analysis_1_spark = (
    prepared
    .groupBy("stock_code")
    .agg(F.count("*").alias("count"))
    .orderBy(
        F.col("count").desc(),
        F.col("stock_code").asc()
    )
)

analysis_1_pdf = analysis_1_spark.toPandas()

# Save the complete frequency table.
analysis_1_pdf.to_csv(
    OUT_DIR / "analysis_1.csv",
    index=False
)

print("Wrote:", OUT_DIR / "analysis_1.csv")
print("Number of stock codes:", len(analysis_1_pdf))
print("Top 20 stock codes:")
print(analysis_1_pdf.head(20).to_string(index=False))

# Verify that the ordering is correct.
top20 = analysis_1_pdf.head(20)

assert all(
    top20.iloc[i]["count"] >= top20.iloc[i + 1]["count"]
    for i in range(len(top20) - 1)
)

print("Analysis 1 ordering verified.")

# Compare with PA2's Top 20 (full data only; the 1,000-row sample has different counts).
PA2_TOP20 = [
    ("85123A", 5829), ("22423", 4424), ("85099B", 4216), ("21212", 3318), ("20725", 3259),
    ("84879", 2960), ("47566", 2768), ("21232", 2747), ("22197", 2549), ("22383", 2540),
    ("20727", 2529), ("21931", 2434), ("22386", 2347), ("22469", 2325), ("22411", 2297),
    ("84991", 2271), ("22382", 2251), ("22384", 2230), ("21080", 2224), ("22086", 2217),
]

if not USING_SAMPLE:
    spark_top20 = list(zip(top20["stock_code"], top20["count"].astype(int)))
    assert len(analysis_1_pdf) == 5305, f"Expected 5,305 stock codes, got {len(analysis_1_pdf)}"
    assert spark_top20 == PA2_TOP20, "Top 20 does not match PA2"
    print("Top 20 matches PA2 exactly (stock code and count); 5,305 stock codes in total.")
else:
    print("Sample run: PA2 Top 20 comparison skipped (full data only).")

Wrote: /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/out/analysis_1.csv
Number of stock codes: 5305
Top 20 stock codes:
stock_code  count
    85123A   5829
     22423   4424
    85099B   4216
     21212   3318
     20725   3259
     84879   2960
     47566   2768
     21232   2747
     22197   2549
     22383   2540
     20727   2529
     21931   2434
     22386   2347
     22469   2325
     22411   2297
     84991   2271
     22382   2251
     22384   2230
     21080   2224
     22086   2217
Analysis 1 ordering verified.
Top 20 matches PA2 exactly (stock code and count); 5,305 stock codes in total.


## 10. Analysis 2 - top three stock codes per country (window)

Finds the three most frequent stock codes in each country, by row count, using the same rows as Analysis 1.

- **Null rule:** rows with a null `country` or `stock_code` are excluded. Section 3a shows there are 0 such rows,
  so this filter removes nothing on this data. `Unspecified` is a real country value and keeps its own top 3.
- **Partitioning:** `Window.partitionBy("country")` ranks each country separately.
- **Ordering:** inside each country, `count` descending puts the most frequent stock codes first.
- **Tie rule:** `row_number()` gives each row exactly one rank, so a tie in `count` is broken by `stock_code`
  ascending (the same rule as Analysis 1). Each country therefore gets at most 3 rows, ranked 1, 2, 3 with no gaps.
- **Why a window:** `groupBy` alone can count stock codes per country, but it cannot rank rows *within* each
  country and keep the row-level detail. A window function adds a rank to every (country, stock code) row
  without collapsing them, so we can then keep ranks 1-3.
- Output: `analysis_2.csv` (columns `country, stock_code, count, rank`).


In [14]:
# Analysis 2: Find the top three stock codes within each country.

# Remove rows with missing country or stock_code.
country_stock_counts = (
    prepared
    .filter(
        F.col("country").isNotNull()
        & F.col("stock_code").isNotNull()
    )
    .groupBy("country", "stock_code")
    .agg(F.count("*").alias("count"))
)

# Create a window for each country.
country_window = Window.partitionBy("country").orderBy(
    F.col("count").desc(),
    F.col("stock_code").asc()
)

analysis_2_spark = (
    country_stock_counts
    .withColumn("rank", F.row_number().over(country_window))
    .filter(F.col("rank") <= 3)
    .select("country", "stock_code", "count", "rank")
    .orderBy(
        F.col("country").asc(),
        F.col("rank").asc(),
        F.col("stock_code").asc()
    )
)

analysis_2_pdf = analysis_2_spark.toPandas()

analysis_2_pdf.to_csv(
    OUT_DIR / "analysis_2.csv",
    index=False
)

print("Wrote:", OUT_DIR / "analysis_2.csv")
print(analysis_2_pdf.head(20).to_string(index=False))

# Confirm that no country has more than three results.
rows_per_country = analysis_2_pdf.groupby("country").size()
assert rows_per_country.max() <= 3

# Ranks in each country must be 1, 2, 3 with no gaps.
for country, part in analysis_2_pdf.groupby("country"):
    assert list(part["rank"]) == list(range(1, len(part) + 1)), country

print("Analysis 2 verified: at most three stock codes per country, ranked with no gaps.")

# Show the country used for the hand check below.
bermuda = analysis_2_pdf[analysis_2_pdf["country"] == "Bermuda"]
print(bermuda.to_string(index=False))
if not USING_SAMPLE:
    assert list(bermuda["stock_code"]) == ["16156S", "20702", "20703"], "Bermuda does not match the hand check"
    print("Bermuda matches the hand check on the TSV.")

Wrote: /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/out/analysis_2.csv
  country stock_code  count  rank
Australia      48138     17     1
Australia      20725     14     2
Australia      21731     13     3
  Austria       POST     34     1
  Austria      22326      8     2
  Austria      20676      7     3
  Bahrain     72807B      4     1
  Bahrain     72802B      3     2
  Bahrain     18097C      2     3
  Belgium       POST    148     1
  Belgium      22326     49     2
  Belgium      22629     36     3
  Bermuda     16156S      1     1
  Bermuda      20702      1     2
  Bermuda      20703      1     3
   Brazil      21166      2     1
   Brazil      22336      2     2
   Brazil      22630      2     3
   Canada      10133      2     1
   Canada      21232      2     2
Analysis 2 verified: at most three stock codes per country, ranked with no gaps.
country stock_code  count  rank
Bermuda     16156S      1     1
Bermuda      20702      1     2
Bermuda     

### 10a. Hand check: Bermuda

Bermuda is a small country (34 rows in the full TSV). Every one of its stock codes appears exactly once, so its
top 3 also tests the tie rule: with all counts equal to 1, the three smallest stock codes (in byte order) win.

Command run from the repository root, with the full TSV in `../data/` (adjust the path to where your copy is;
field 2 is `StockCode`, field 8 is `Country`):

```bash
awk -F'\t' 'NR>1 && $8=="Bermuda" {print $2}' ../data/online_retail_II_combined.tsv \
  | LC_ALL=C sort | uniq -c | LC_ALL=C sort -k1,1nr -k2,2 | head -3
```

Result:

```text
   1 16156S
   1 20702
   1 20703
```

This matches the PySpark result above (ranks 1-3: `16156S`, `20702`, `20703`, each with count 1).


## 11. Analysis 1 in DuckDB and row-by-row comparison

Runs the same Analysis 1 query in DuckDB on the **same Parquet files** that Spark reads, then compares the two results.

- The two results are compared on **every row** by an outer merge on `stock_code`, not by position, so a different
  tie order could never cause a false mismatch.
- **Tolerance: 0.** The counts are integers, so the two engines must agree exactly. A stock code found by only one
  engine also counts as a mismatch.
- Any mismatches are written to `analysis_1_mismatches.csv` (header only when there are none).
- `results_match` stores the overall pass/fail; Section 12 uses it for `result_verified`.


In [15]:
parquet_glob = str(PARQUET_DIR / "*.parquet")

# Reproduce Analysis 1 in DuckDB.
duckdb_query = f"""
    SELECT stock_code, COUNT(*) AS count
    FROM read_parquet('{parquet_glob}')
    GROUP BY stock_code
    ORDER BY count DESC, stock_code ASC
"""

duckdb_analysis_1_pdf = duckdb.sql(duckdb_query).df()

print("DuckDB result:")
print(duckdb_analysis_1_pdf.head(20).to_string(index=False))

# Make sure both tables have the same column names and data types.
spark_result = analysis_1_pdf.copy()
duck_result = duckdb_analysis_1_pdf.copy()

spark_result["stock_code"] = spark_result["stock_code"].astype("string")
duck_result["stock_code"] = duck_result["stock_code"].astype("string")

spark_result["count"] = spark_result["count"].astype("int64")
duck_result["count"] = duck_result["count"].astype("int64")

# Compare every result row.
comparison = spark_result.merge(
    duck_result,
    on="stock_code",
    how="outer",
    suffixes=("_spark", "_duckdb"),
    indicator=True
)

comparison["count_spark"] = comparison["count_spark"].fillna(-1)
comparison["count_duckdb"] = comparison["count_duckdb"].fillna(-1)

comparison["count_difference"] = (
    comparison["count_spark"] - comparison["count_duckdb"]
).abs()

# Counts are integers, so the tolerance is zero.
TOLERANCE = 0

mismatches = comparison[
    (comparison["_merge"] != "both")
    | (comparison["count_difference"] > TOLERANCE)
].copy()

mismatches.to_csv(
    OUT_DIR / "analysis_1_mismatches.csv",
    index=False
)

results_match = mismatches.empty

print("Spark result rows:", len(spark_result))
print("DuckDB result rows:", len(duck_result))
print("Only in Spark:", int((comparison["_merge"] == "left_only").sum()))
print("Only in DuckDB:", int((comparison["_merge"] == "right_only").sum()))
print("Rows with different counts:", int(((comparison["_merge"] == "both") & (comparison["count_difference"] > TOLERANCE)).sum()))
print("Mismatches:", len(mismatches))
print("Results match:", results_match)

assert results_match, "PySpark and DuckDB results do not match."

print("PySpark and DuckDB Analysis 1 results match exactly.")

DuckDB result:
stock_code  count
    85123A   5829
     22423   4424
    85099B   4216
     21212   3318
     20725   3259
     84879   2960
     47566   2768
     21232   2747
     22197   2549
     22383   2540
     20727   2529
     21931   2434
     22386   2347
     22469   2325
     22411   2297
     84991   2271
     22382   2251
     22384   2230
     21080   2224
     22086   2217
Spark result rows: 5305
DuckDB result rows: 5305
Only in Spark: 0
Only in DuckDB: 0
Rows with different counts: 0
Mismatches: 0
Results match: True
PySpark and DuckDB Analysis 1 results match exactly.


## 12. Benchmark

Times one PySpark run and one DuckDB run of Analysis 1 and writes `benchmark.csv` with exactly this header:
`implementation,environment,input_rows,input_bytes,format,elapsed_seconds,output_rows,result_verified`

**How the timing is measured (the same for both engines):**

- One timed run each with `time.perf_counter()`. No repeated runs and no averages.
- The timer covers reading the Parquet, the Analysis 1 aggregation, and returning the result to Python:
  `.collect()` for Spark (Spark is lazy, so nothing runs without an action) and `.fetchall()` for DuckDB.
- It does **not** include starting the SparkSession or writing CSV files. Nothing is cached before timing.
- Both engines read the same Parquet files, so `input_rows` and `input_bytes` are the same for both.
- `result_verified` is `True` only if the Section 11 comparison found no mismatches.

**Environment:** one machine. Spark runs in `local[*]` mode (driver and executors in one JVM on this machine);
DuckDB runs inside the Python process. The `environment` column records the engine version, OS, and core count.

**Caveats:** PySpark runs first and DuckDB second. Both engines already ran this query in Sections 9 and 11, so
Spark's JVM is warmed up and the Parquet files are probably in the operating system's file cache. A single run on
small data mostly measures fixed overhead (Spark job scheduling vs. DuckDB's in-process execution), so the numbers
should not be read as how the engines compare on large data.


In [16]:
import time

# Describe the machine for the environment column.
machine = f"{platform.system()} {platform.machine()}, {os.cpu_count()} cores"
spark_environment = f"Spark {spark.version} {spark.sparkContext.master}, {machine}"
duckdb_environment = f"DuckDB {duckdb.__version__} in-process, same machine ({machine})"

# Calculate the Parquet input size.
input_bytes = sum(
    path.stat().st_size
    for path in PARQUET_DIR.rglob("*.parquet")
)

input_rows = prepared.count()

# -----------------------------
# Timed PySpark Analysis 1 run
# -----------------------------
spark_start = time.perf_counter()

spark_benchmark_result = (
    spark.read
    .parquet(str(PARQUET_DIR))
    .groupBy("stock_code")
    .agg(F.count("*").alias("count"))
    .orderBy(
        F.col("count").desc(),
        F.col("stock_code").asc()
    )
    .collect()
)

spark_elapsed = time.perf_counter() - spark_start
spark_output_rows = len(spark_benchmark_result)

# -----------------------------
# Timed DuckDB Analysis 1 run
# -----------------------------
duckdb_start = time.perf_counter()

duckdb_benchmark_result = duckdb.sql(duckdb_query).fetchall()

duckdb_elapsed = time.perf_counter() - duckdb_start
duckdb_output_rows = len(duckdb_benchmark_result)

# -----------------------------
# Create benchmark table
# -----------------------------
benchmark_pdf = pd.DataFrame([
    {
        "implementation": "PySpark",
        "environment": spark_environment,
        "input_rows": input_rows,
        "input_bytes": input_bytes,
        "format": "Parquet",
        "elapsed_seconds": round(spark_elapsed, 6),
        "output_rows": spark_output_rows,
        "result_verified": bool(results_match),
    },
    {
        "implementation": "DuckDB",
        "environment": duckdb_environment,
        "input_rows": input_rows,
        "input_bytes": input_bytes,
        "format": "Parquet",
        "elapsed_seconds": round(duckdb_elapsed, 6),
        "output_rows": duckdb_output_rows,
        "result_verified": bool(results_match),
    },
])

benchmark_pdf.to_csv(
    OUT_DIR / "benchmark.csv",
    index=False
)

print("Wrote:", OUT_DIR / "benchmark.csv")
print(benchmark_pdf.to_string(index=False))

Wrote: /Users/kanishka/CS_131/project/team9-online-retail-recommendations/pa3/out/benchmark.csv
implementation                                                    environment  input_rows  input_bytes  format  elapsed_seconds  output_rows  result_verified
       PySpark                   Spark 4.3.0 local[*], Darwin arm64, 10 cores     1067371      9092629 Parquet         0.511896         5305             True
        DuckDB DuckDB 2.0.1 in-process, same machine (Darwin arm64, 10 cores)     1067371      9092629 Parquet         0.005357         5305             True


## 13. Chart

**Kushan:** To be completed.

## 14. Findings and limitations

**Kushan:** To be completed.

## 15. AI-use disclosure

- **Kanishka Yadav (Data Engineer):** Used Claude to understand the PA3 requirements and how the PySpark, Parquet, and DuckDB steps fit together, and to check that the notebook's code, explanations, and outputs are consistent. Reviewed all of it, ran the notebook on the full dataset, and confirmed the results against PA2.
- **Sanjini Shukla (Data Engineer):** Used Codex to understand what Sections 9-12 required and how grouped aggregations, window functions, and DuckDB queries work in this project. Reviewed and ran the code and checked the outputs.